# Contextual Retrieval — A Step-by-Step Learning Notebook

Based on Anthropic's Engineering Blog: *Introducing Contextual Retrieval* (Sep 2024)

---

## What We Will Build

We will go through every technique mentioned in the blog, in order:

1. **Standard Embedding RAG** — the baseline
2. **BM25** — lexical (keyword) search
3. **Hybrid RAG** — combining embeddings + BM25
4. **Contextual Retrieval** — adding AI-generated context to chunks
5. **Contextual Embeddings** — embedding the enriched chunks
6. **Contextual BM25** — keyword search on enriched chunks
7. **Reranking** — final scoring pass to pick the best chunks
8. **Full Pipeline** — everything combined

---

## The Core Problem We Are Solving

When you split a document into chunks, individual chunks often lose their context.

For example, a chunk might say:
> *"The company's revenue grew by 3% over the previous quarter."*

But which company? Which quarter? The chunk on its own is useless.

Contextual Retrieval fixes this by prepending a short AI-generated description to each chunk before indexing it.

## Step 0 — Install Dependencies

In [1]:
!pip install anthropic sentence-transformers rank-bm25 numpy --quiet

## Step 1 — Set Up & Load a Sample Document

We will use a short fictional SEC filing document as our knowledge base — similar to the example in the blog.

In [ ]:
from anthropic import Anthropic
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import fitz  # PyMuPDF
import numpy as np
from pathlib import Path
from anthropic import Anthropic
import voyageai

# ── Your Anthropic API key ─────────────────────────────────────────
client = Anthropic()
# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — PDF Text Extraction
# ─────────────────────────────────────────────────────────────────────────────

def extract_text_from_pdf(pdf_path: str) -> list[dict]:
    """
    Extracts text from each page of a PDF.

    Fixes applied:
      - Raises early if no text is found (scanned/image PDF guard)
      - Reports which pages returned empty text (for debugging)
    """
    doc = fitz.open(pdf_path)
    pages = []
    empty_pages = []

    for i, page in enumerate(doc):
        text = page.get_text("text").strip()
        if text:
            pages.append({"page_num": i + 1, "text": text})
        else:
            empty_pages.append(i + 1)

    doc.close()

    if empty_pages:
        print(f"  ⚠️  Pages with no extractable text: {empty_pages}")

    # Hard stop for fully scanned PDFs — silent empty list causes confusing
    # downstream errors (embeddings on [] etc.)
    if not pages:
        raise ValueError(
            f"No extractable text found in '{pdf_path}'.\n"
            "The file may be a scanned/image-based PDF. "
            "Consider pre-processing with OCR (e.g. pytesseract or AWS Textract)."
        )

    return pages


# ── 2. Split into sentences (lightweight, no NLTK needed) ────────────────────
import re

def split_into_sentences(text: str) -> list[str]:
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s.strip()]


# ── 7. Run it over your folder ────────────────────────────────────────────────
def process_pdf_folder(
    folder: str = "./reg_research_paper",
    add_context: bool = True,            # set False to skip Claude enrichment
) -> dict[str, list[dict]]:

    results = {}
    for pdf_file in Path(folder).glob("*.pdf"):
        print(f"\nProcessing: {pdf_file.name}")
        chunks = semantic_chunk_pdf(str(pdf_file))
        print(f"  → {len(chunks)} semantic chunks created")

        if add_context:
            # pass full text for contextualisation
            full_text = " ".join(c["text"] for c in chunks)
            print(f"  Adding contextual retrieval context…")
            chunks = add_contextual_retrieval(chunks, full_text)

        results[pdf_file.stem] = chunks
    return results


c:\Users\Parth\Downloads\Agentic_Rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Document loaded. Length: 1156 characters


## Step 2 — Chunk the Document

RAG starts by splitting the document into smaller chunks. The blog recommends chunks of a few hundred tokens each.

In [3]:
def split_into_chunks(text, chunk_size=200):
    """
    Simple sentence-aware chunker.
    In production you would use a more sophisticated splitter
    with overlap between chunks.
    """
    sentences = [s.strip() for s in text.split('.') if s.strip()]
    chunks = []
    current_chunk = []
    current_len = 0

    for sentence in sentences:
        if current_len + len(sentence) > chunk_size and current_chunk:
            chunks.append('. '.join(current_chunk) + '.')
            current_chunk = [sentence]
            current_len = len(sentence)
        else:
            current_chunk.append(sentence)
            current_len += len(sentence)

    if current_chunk:
        chunks.append('. '.join(current_chunk) + '.')

    return chunks


chunks = split_into_chunks(DOCUMENT)

print(f"Created {len(chunks)} chunks:\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk)
    print()

Created 8 chunks:

--- Chunk 1 ---
ACME Corp — Q2 2023 SEC Filing

Company Overview
ACME Corp is a mid-size technology company headquartered in Austin, Texas.

--- Chunk 2 ---
It provides cloud infrastructure services to enterprise clients across North America. The company was founded in 2010 and went public in 2018.

--- Chunk 3 ---
Q1 2023 Financial Results
In Q1 2023, ACME Corp reported revenue of $314 million, representing a 12% 
year-over-year increase. Operating expenses were $280 million.

--- Chunk 4 ---
Net income was 
$34 million, up from $28 million in Q1 2022. Q2 2023 Financial Results
The company's revenue grew by 3% over the previous quarter. Total revenue reached $323 million.

--- Chunk 5 ---
Operating costs increased slightly to $285 million
due to new data center expansions in Phoenix and Atlanta. Net income for Q2 2023 was $38 million.

--- Chunk 6 ---
Customer Growth
The number of enterprise clients grew from 1,240 in Q1 to 1,310 in Q2 2023. Customer churn remained 

### Notice the problem

Look at the chunk that says:
> *"The company's revenue grew by 3% over the previous quarter."*

Without context, we don't know:
- Which company?
- Which quarter?
- What was the previous quarter's revenue?

This is exactly the problem Contextual Retrieval solves.

## Step 3 — Standard Embedding RAG (Baseline)

We embed each chunk into a vector, then at query time find the most similar chunks using cosine similarity.

In [4]:
# Load a lightweight embedding model
# In production, Anthropic's blog found Voyage and Gemini embeddings perform best
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

def embed(texts):
    return embedding_model.encode(texts, normalize_embeddings=True)

def cosine_similarity(a, b):
    return np.dot(a, b)  # already normalized so dot = cosine

def embedding_search(query, chunks, chunk_embeddings, top_k=3):
    query_embedding = embed([query])[0]
    scores = [cosine_similarity(query_embedding, ce) for ce in chunk_embeddings]
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    return [(chunks[i], score) for i, score in ranked[:top_k]]


# Embed all chunks
print("Embedding chunks...")
chunk_embeddings = embed(chunks)
print(f"Embedded {len(chunks)} chunks. Each vector has {chunk_embeddings.shape[1]} dimensions.\n")

# Test query
query = "What was the revenue growth for ACME Corp in Q2 2023?"
results = embedding_search(query, chunks, chunk_embeddings)

print(f"Query: '{query}'\n")
print("Top results (Standard Embeddings):")
for i, (chunk, score) in enumerate(results):
    print(f"\n[{i+1}] Score: {score:.4f}")
    print(chunk)

Embedding chunks...
Embedded 8 chunks. Each vector has 384 dimensions.

Query: 'What was the revenue growth for ACME Corp in Q2 2023?'

Top results (Standard Embeddings):

[1] Score: 0.8577
Q1 2023 Financial Results
In Q1 2023, ACME Corp reported revenue of $314 million, representing a 12% 
year-over-year increase. Operating expenses were $280 million.

[2] Score: 0.6561
ACME Corp — Q2 2023 SEC Filing

Company Overview
ACME Corp is a mid-size technology company headquartered in Austin, Texas.

[3] Score: 0.6327
Net income was 
$34 million, up from $28 million in Q1 2022. Q2 2023 Financial Results
The company's revenue grew by 3% over the previous quarter. Total revenue reached $323 million.


## Step 4 — BM25 (Keyword / Lexical Search)

BM25 is a classic keyword search algorithm. Unlike embeddings, it looks for exact word matches. It is especially good for queries with specific terms, codes, or proper nouns that embeddings might miss.

**How BM25 works:**
- Counts how often query words appear in each chunk (TF — Term Frequency)
- Penalises words that appear in many chunks, making rare words more valuable (IDF — Inverse Document Frequency)
- Applies a length normalisation so longer chunks don't automatically score higher

## 1. `build_bm25_index`

```python
def build_bm25_index(chunks):
    tokenized = [chunk.lower().split() for chunk in chunks]
    return BM25Okapi(tokenized)
```

**What it does in plain English:**

It converts every chunk from a sentence into a list of individual words, then hands that to BM25Okapi which builds a statistical table in memory.

**Example — imagine you have 3 chunks:**

```
Chunk 0: "ACME Corp revenue grew by 3%"
Chunk 1: "The company expanded into Europe"
Chunk 2: "Revenue reached 323 million"
```

After `.lower().split()`, tokenized looks like:

```
[
  ["acme", "corp", "revenue", "grew", "by", "3%"],
  ["the", "company", "expanded", "into", "europe"],
  ["revenue", "reached", "323", "million"]
]
```

`BM25Okapi` then builds an internal table that tracks, for every unique word across all chunks:
- How many times does it appear in each chunk (TF)
- How many chunks contain it at all (for IDF)
- How long each chunk is (for length normalisation)

So after building the index, BM25 already knows internally:
- "revenue" → appears in chunk 0 and chunk 2 (common → less valuable)
- "acme" → appears only in chunk 0 (rare → more valuable)
- "europe" → appears only in chunk 1 (rare → more valuable)

**This index is built once and reused for every query.** You never rebuild it unless your chunks change.

---

## 2. `bm25_search`

```python
def bm25_search(query, chunks, bm25_index, top_k=3):
    tokenized_query = query.lower().split()
    scores = bm25_index.get_scores(tokenized_query)
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    return [(chunks[i], score) for i, score in ranked[:top_k]]
```

Let's say the query is:

```
"What was the revenue growth for ACME Corp?"
```

**Line by line:**

**Line 1 — tokenize the query:**
```
tokenized_query = ["what", "was", "the", "revenue", "growth", "for", "acme", "corp?"]
```
Same treatment as the chunks — lowercase and split into words.

**Line 2 — `get_scores()`:**

This is where all the BM25 math happens. For each word in the query, BM25 looks it up in the index and scores every chunk. Let's trace "revenue" and "acme" as examples:

**"revenue":**
- Appears in chunk 0 once, chunk 2 once
- It's in 2 out of 3 chunks → not that rare → IDF is low → contributes a modest score to chunks 0 and 2

**"acme":**
- Appears only in chunk 0
- It's in 1 out of 3 chunks → quite rare → IDF is high → contributes a big score to chunk 0

**"growth":**
- Appears in 0 chunks
- Score contribution = 0 everywhere

After summing all query words, `get_scores()` returns one score per chunk:

```
scores = [4.2, 0.0, 1.1]
          ^          ^
        chunk 0    chunk 2
        (has both  (has revenue
        revenue    but not acme)
        and acme)
```

**Line 3 — sort:**
```
ranked = [(0, 4.2), (2, 1.1), (1, 0.0)]
```
Sorted highest score first.

**Line 4 — return top_k:**
Returns the actual chunk text alongside its score.

---

## Your Question — How Does TF Actually Work in the Code?

You never explicitly write a TF counter — `BM25Okapi` handles it internally when you call `build_bm25_index`. The moment you pass in the tokenized chunks, it counts term frequencies for every word in every chunk and stores them.

But BM25 doesn't use raw TF. It applies a **saturation function** on top:

```
TF_saturated = (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * doc_len / avg_doc_len))
```

What this means practically: if "revenue" appears 1 time in a chunk, it gets a decent score. If it appears 10 times, it gets a higher score — but not 10x higher. The score *saturates*. This stops a chunk that just repeats the same word 50 times from dominating the results.

---

## Your Question — Where Does the Penalisation of Common Words Happen?

That's the IDF part, and again `BM25Okapi` computes it automatically during `build_bm25_index`. The formula is:

```
IDF(word) = log((N - df + 0.5) / (df + 0.5))
```

Where:
- `N` = total number of chunks (3 in our example)
- `df` = number of chunks the word appears in

So for "revenue" appearing in 2 of 3 chunks:
```
IDF("revenue") = log((3 - 2 + 0.5) / (2 + 0.5)) = log(1.5 / 2.5) = log(0.6) ≈ -0.5
```
Low (even slightly negative) — common word, not useful for distinguishing chunks.

For "acme" appearing in 1 of 3 chunks:
```
IDF("acme") = log((3 - 1 + 0.5) / (1 + 0.5)) = log(2.5 / 1.5) = log(1.67) ≈ 0.51
```
Positive — rare word, very useful for finding the right chunk.

The final BM25 score for each chunk is just `TF_saturated × IDF` summed across all query words. You never write any of this math yourself — it's all inside `BM25Okapi`. Your only job is to feed it clean tokenized text.

In [6]:
def build_bm25_index(chunks):
    tokenized = [chunk.lower().split() for chunk in chunks]
    return BM25Okapi(tokenized)

def bm25_search(query, chunks, bm25_index, top_k=3):
    tokenized_query = query.lower().split()
    scores = bm25_index.get_scores(tokenized_query)
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    return [(chunks[i], score) for i, score in ranked[:top_k]]


bm25_index = build_bm25_index(chunks)

results_bm25 = bm25_search(query, chunks, bm25_index)

print(f"Query: '{query}'\n")
print("Top results (BM25):")
for i, (chunk, score) in enumerate(results_bm25):
    print(f"\n[{i+1}] Score: {score:.4f}")
    print(chunk)

Query: 'My name is Parth'

Top results (BM25):

[1] Score: 1.7288
ACME Corp — Q2 2023 SEC Filing

Company Overview
ACME Corp is a mid-size technology company headquartered in Austin, Texas.

[2] Score: 0.0000
It provides cloud infrastructure services to enterprise clients across North America. The company was founded in 2010 and went public in 2018.

[3] Score: 0.0000
Q1 2023 Financial Results
In Q1 2023, ACME Corp reported revenue of $314 million, representing a 12% 
year-over-year increase. Operating expenses were $280 million.


In [7]:
query= "My name is Parth"
query.lower().split()

['my', 'name', 'is', 'parth']

## Step 5 — Hybrid Search (Embeddings + BM25)

The blog recommends combining both methods using **Reciprocal Rank Fusion (RRF)**. This merges the rankings from both approaches — each result gets a score based on its position in each ranked list, and the scores are summed.

In [ ]:
def reciprocal_rank_fusion(results_list, k=60):
    """
    Combine multiple ranked lists into one using RRF.
    k=60 is the standard constant recommended in the original RRF paper.
    """
    scores = {}
    for results in results_list:
        for rank, (chunk, _) in enumerate(results):
            if chunk not in scores:
                scores[chunk] = 0
            scores[chunk] += 1 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


# Get more results from each method before fusing
emb_results  = embedding_search(query, chunks, chunk_embeddings, top_k=len(chunks))
bm25_results = bm25_search(query, chunks, bm25_index, top_k=len(chunks))

hybrid_results = reciprocal_rank_fusion([emb_results, bm25_results])

print(f"Query: '{query}'\n")
print("Top results (Hybrid — Embeddings + BM25):")
for i, (chunk, score) in enumerate(hybrid_results[:3]):
    print(f"\n[{i+1}] RRF Score: {score:.4f}")
    print(chunk)

## Step 6 — Contextual Retrieval (The Key Innovation)

This is the main contribution of the blog post. Before indexing, we use Claude to generate a short context description for each chunk, then prepend it to the chunk.

The prompt from the blog:

```
<document>
{{WHOLE_DOCUMENT}}
</document>
Here is the chunk we want to situate within the whole document:
<chunk>
{{CHUNK_CONTENT}}
</chunk>
Please give a short succinct context to situate this chunk within the overall document
for the purposes of improving search retrieval of the chunk.
Answer only with the succinct context and nothing else.
```

The blog also recommends using **prompt caching** so you only send the full document to the API once per batch, not once per chunk — making this very cost-efficient.


**The key rules for cache efficiency:**

1. **The cached content must be identical across calls** — even one character difference creates a new cache entry
2. **Cached content must come before non-cached content** in the message — the chunk goes after the `cache_control` block, never before
3. **Finish all chunks of one document before starting the next** — switching documents mid-way wastes the warm cache
4. **The cache expires after 5 minutes of inactivity** — if your chunk processing is slow, the cache may expire before you finish a large document. Process in batches if needed.

Want me to update the notebook with this corrected implementation?

In [ ]:
def generate_context_for_chunk(document, chunk, client):
    """
    Uses Claude to generate a short context description for a chunk.
    
    In production, use prompt caching by marking the document with
    cache_control so it is only processed once across all chunks.
    This reduces cost to ~$1.02 per million document tokens.
    """
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",  # Use Haiku — fast and cheap for this task
        max_tokens=150,
        messages=[
            {
                "role": "user",
                "content": (
                    f"<document>\n{document}\n</document>\n\n"
                    f"Here is the chunk we want to situate within the whole document:\n"
                    f"<chunk>\n{chunk}\n</chunk>\n\n"
                    f"Please give a short succinct context to situate this chunk within "
                    f"the overall document for the purposes of improving search retrieval "
                    f"of the chunk. Answer only with the succinct context and nothing else."
                )
            }
        ]
    )
    return response.content[0].text.strip()


def build_contextual_chunks(document, chunks, client):
    """
    For each chunk: generate context → prepend it → return enriched chunk.
    The blog says the context is typically 50-100 tokens.
    """
    contextual_chunks = []
    for i, chunk in enumerate(chunks):
        print(f"Generating context for chunk {i+1}/{len(chunks)}...")
        context = generate_context_for_chunk(document, chunk, client)
        enriched = f"{context}\n\n{chunk}"
        contextual_chunks.append(enriched)
        print(f"  Context: {context[:100]}..." if len(context) > 100 else f"  Context: {context}")
    return contextual_chunks


# NOTE: This will call the Anthropic API for each chunk
# Make sure your API key is set before running this cell
contextual_chunks = build_contextual_chunks(DOCUMENT, chunks, client)

print("\n\n=== Example: Before vs After Contextualisation ===")
print("\nORIGINAL CHUNK:")
print(chunks[2])  # The ambiguous revenue chunk
print("\nCONTEXTUALISED CHUNK:")
print(contextual_chunks[2])

In [ ]:
def generate_context_for_chunk_cached(document, chunk, client):
    """
    Uses Claude to generate context with prompt caching.
    
    The document is marked with cache_control so it is stored
    server-side after the first call. All subsequent chunks from
    the SAME document hit the cache instead of reprocessing.
    
    Cost: first call = full price, subsequent calls = ~10% of price.
    The cache persists for 5 minutes of inactivity.
    """
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=150,
        system=[
            {
                "type": "text",
                "text": "You are a helpful assistant that generates concise context for document chunks.",
                # Mark the system prompt for caching too if it's long
            }
        ],
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        # The document is the expensive part — cache it
                        "type": "text",
                        "text": f"<document>\n{document}\n</document>\n\n",
                        "cache_control": {"type": "ephemeral"}  # <-- THIS is what was missing
                    },
                    {
                        # The chunk-specific part is NOT cached — it changes every call
                        "type": "text",
                        "text": (
                            f"Here is the chunk we want to situate within the whole document:\n"
                            f"<chunk>\n{chunk}\n</chunk>\n\n"
                            f"Please give a short succinct context to situate this chunk within "
                            f"the overall document for the purposes of improving search retrieval "
                            f"of the chunk. Answer only with the succinct context and nothing else."
                        )
                    }
                ]
            }
        ]
    )
    
    # You can inspect cache usage in the response
    usage = response.usage
    print(f"  Tokens — input: {usage.input_tokens}, "
          f"cache_read: {getattr(usage, 'cache_read_input_tokens', 0)}, "
          f"cache_write: {getattr(usage, 'cache_creation_input_tokens', 0)}")
    
    return response.content[0].text.strip()


def build_contextual_chunks_with_cache(documents_and_chunks, client):
    """
    documents_and_chunks: list of (document_text, [chunk1, chunk2, ...])
    
    Processes ALL chunks of each document together before moving
    to the next document — this is critical for cache efficiency.
    Never interleave chunks across different documents.
    """
    all_contextual_chunks = []
    
    for doc_idx, (document, chunks) in enumerate(documents_and_chunks):
        print(f"\nDocument {doc_idx + 1}/{len(documents_and_chunks)}")
        print(f"Processing {len(chunks)} chunks — document will be cached after first call\n")
        
        doc_contextual_chunks = []
        for i, chunk in enumerate(chunks):
            print(f"  Chunk {i+1}/{len(chunks)}:")
            context = generate_context_for_chunk_cached(document, chunk, client)
            enriched = f"{context}\n\n{chunk}"
            doc_contextual_chunks.append(enriched)
        
        all_contextual_chunks.extend(doc_contextual_chunks)
    
    return all_contextual_chunks


# Usage example with multiple documents:
# documents_and_chunks = [
#     (document_A_text, chunks_of_A),
#     (document_B_text, chunks_of_B),
#     (document_C_text, chunks_of_C),
# ]
# contextual_chunks = build_contextual_chunks_with_cache(documents_and_chunks, client)

## Step 7 — Contextual Embeddings

Now we embed the enriched (contextualised) chunks instead of the raw chunks. The embedding model now has the context to work with, so similar queries will match more reliably.

In [ ]:
print("Embedding contextual chunks...")
contextual_embeddings = embed(contextual_chunks)
print(f"Done. Shape: {contextual_embeddings.shape}\n")

# Search with contextual embeddings
contextual_emb_results = embedding_search(
    query, contextual_chunks, contextual_embeddings
)

print(f"Query: '{query}'\n")
print("Top results (Contextual Embeddings):")
for i, (chunk, score) in enumerate(contextual_emb_results):
    print(f"\n[{i+1}] Score: {score:.4f}")
    print(chunk)

## Step 8 — Contextual BM25

We also rebuild the BM25 index using the contextualised chunks. Now keyword search also benefits from the added context — terms like "ACME Corp" and "Q2 2023" appear in chunks that previously lacked them.

In [ ]:
contextual_bm25_index = build_bm25_index(contextual_chunks)

contextual_bm25_results = bm25_search(
    query, contextual_chunks, contextual_bm25_index
)

print(f"Query: '{query}'\n")
print("Top results (Contextual BM25):")
for i, (chunk, score) in enumerate(contextual_bm25_results):
    print(f"\n[{i+1}] Score: {score:.4f}")
    print(chunk)

## Step 9 — Reranking

The final step from the blog. After getting the top N candidates from hybrid search, a reranker model scores each (query, chunk) pair together — not as separate vectors, but as a combined input. This is slower but far more accurate.

**In production:** Anthropic used the Cohere reranker. Voyage also offers one. Here we simulate the behaviour using cross-encoder scoring from sentence-transformers.

**The blog's recommended flow:**
1. Retrieve top 150 candidates via hybrid search
2. Rerank them → keep top 20
3. Pass the top 20 to the LLM as context

In [ ]:
def embedding_search(query, chunks, chunk_embeddings, top_k=3):
    query_embedding = embed([query])[0]
    scores = [cosine_similarity(query_embedding, ce) for ce in chunk_embeddings]
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    return [(chunks[i], score) for i, score in ranked[:top_k]]

In [ ]:
from sentence_transformers import CrossEncoder

# Load a cross-encoder reranker
# In production use: Cohere reranker API or Voyage reranker API
print("Loading reranker model...")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("Reranker loaded.\n")


def rerank(query, candidates, reranker, top_k=3):
    """
    Score each (query, chunk) pair together using a cross-encoder.
    Unlike embeddings which encode query and chunk separately,
    the cross-encoder reads both texts simultaneously and can
    reason about their relationship directly.
    """
    pairs = [[query, chunk] for chunk in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]


# Get top candidates via hybrid contextual search first
ctx_emb_all  = embedding_search(query, contextual_chunks, contextual_embeddings, top_k=len(contextual_chunks))
ctx_bm25_all = bm25_search(query, contextual_chunks, contextual_bm25_index, top_k=len(contextual_chunks))
hybrid_ctx   = reciprocal_rank_fusion([ctx_emb_all, ctx_bm25_all])

# Take top candidates and rerank them
top_candidates = [chunk for chunk, _ in hybrid_ctx[:min(10, len(hybrid_ctx))]]
reranked_results = rerank(query, top_candidates, reranker)

print(f"Query: '{query}'\n")
print("Top results (Full Pipeline — Contextual Embeddings + Contextual BM25 + Reranking):")
for i, (chunk, score) in enumerate(reranked_results):
    print(f"\n[{i+1}] Reranker Score: {score:.4f}")
    print(chunk)

## Step 10 — Full Pipeline End-to-End

Let's put it all together in a clean function that mirrors exactly what the blog describes.

In [ ]:
def full_contextual_retrieval_pipeline(
    query,
    document,
    contextual_chunks,
    contextual_embeddings,
    contextual_bm25_index,
    reranker,
    top_k_retrieve=10,   # blog uses 150 for large knowledge bases
    top_k_rerank=3       # blog uses 20 for final context window
):
    """
    Full pipeline from the Anthropic engineering blog:
    1. Retrieve candidates via Contextual Embeddings
    2. Retrieve candidates via Contextual BM25
    3. Combine with Reciprocal Rank Fusion
    4. Rerank the top candidates
    5. Return the top-K chunks for the LLM
    """
    # Step 1 & 2: Retrieve
    emb_results  = embedding_search(query, contextual_chunks, contextual_embeddings, top_k=top_k_retrieve)
    bm25_results = bm25_search(query, contextual_chunks, contextual_bm25_index, top_k=top_k_retrieve)

    # Step 3: Fuse
    fused = reciprocal_rank_fusion([emb_results, bm25_results])
    candidates = [chunk for chunk, _ in fused[:top_k_retrieve]]

    # Step 4: Rerank
    reranked = rerank(query, candidates, reranker, top_k=top_k_rerank)

    return reranked


def generate_answer(query, retrieved_chunks, client):
    """
    Pass the retrieved chunks to Claude as context and get an answer.
    """
    context = "\n\n---\n\n".join([chunk for chunk, _ in retrieved_chunks])
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=[
            {
                "role": "user",
                "content": (
                    f"Use the following context to answer the question.\n\n"
                    f"Context:\n{context}\n\n"
                    f"Question: {query}"
                )
            }
        ]
    )
    return response.content[0].text


# Run the full pipeline
print("Running full pipeline...\n")
top_chunks = full_contextual_retrieval_pipeline(
    query=query,
    document=DOCUMENT,
    contextual_chunks=contextual_chunks,
    contextual_embeddings=contextual_embeddings,
    contextual_bm25_index=contextual_bm25_index,
    reranker=reranker
)

print("Retrieved chunks:")
for i, (chunk, score) in enumerate(top_chunks):
    print(f"\n[{i+1}] Score: {score:.4f}")
    print(chunk)

print("\n" + "="*60)
print("Generating final answer...\n")
answer = generate_answer(query, top_chunks, client)
print("ANSWER:")
print(answer)

## Step 11 — Compare All Approaches Side by Side

Let's run the same query through all four methods and compare what each retrieves.

In [ ]:
test_queries = [
    "What was the revenue growth for ACME Corp in Q2 2023?",
    "How many enterprise clients did the company have?",
    "What are the company's plans for Europe?"
]

for query in test_queries:
    print("=" * 70)
    print(f"QUERY: {query}")
    print("=" * 70)

    # Method 1: Standard embeddings only
    r1 = embedding_search(query, chunks, chunk_embeddings, top_k=1)
    print(f"\n[1] Standard Embeddings:")
    print(f"    {r1[0][0][:120]}..." if len(r1[0][0]) > 120 else f"    {r1[0][0]}")

    # Method 2: Embeddings + BM25 (no context)
    r2_emb  = embedding_search(query, chunks, chunk_embeddings, top_k=len(chunks))
    r2_bm25 = bm25_search(query, chunks, bm25_index, top_k=len(chunks))
    r2 = reciprocal_rank_fusion([r2_emb, r2_bm25])
    print(f"\n[2] Hybrid (Embed + BM25, no context):")
    print(f"    {r2[0][0][:120]}..." if len(r2[0][0]) > 120 else f"    {r2[0][0]}")

    # Method 3: Contextual Embeddings + Contextual BM25
    r3_emb  = embedding_search(query, contextual_chunks, contextual_embeddings, top_k=len(contextual_chunks))
    r3_bm25 = bm25_search(query, contextual_chunks, contextual_bm25_index, top_k=len(contextual_chunks))
    r3 = reciprocal_rank_fusion([r3_emb, r3_bm25])
    print(f"\n[3] Contextual Hybrid (cEmbed + cBM25):")
    print(f"    {r3[0][0][:120]}..." if len(r3[0][0]) > 120 else f"    {r3[0][0]}")

    # Method 4: Full pipeline with reranking
    r4 = full_contextual_retrieval_pipeline(
        query, DOCUMENT, contextual_chunks,
        contextual_embeddings, contextual_bm25_index, reranker
    )
    print(f"\n[4] Full Pipeline (cEmbed + cBM25 + Rerank):")
    print(f"    {r4[0][0][:120]}..." if len(r4[0][0]) > 120 else f"    {r4[0][0]}")

    print()

## Summary — What Each Step Adds

| Step | Method | What It Adds |
|------|--------|--------------|
| 1 | Standard Embeddings | Semantic similarity search |
| 2 | + BM25 | Exact keyword matching for specific terms |
| 3 | + Contextual Enrichment | Each chunk knows what document/section it came from |
| 4 | + Contextual BM25 | Keywords from context (company name, dates) now searchable |
| 5 | + Reranking | Cross-encoder re-scores each (query, chunk) pair together |

---

## Key Findings From the Blog (with our critical lens)

- **Contextual enrichment helps most** for ambiguous/narrative content (fiction, enterprise docs, legal memos) where chunks genuinely lose meaning in isolation
- **Reranking does the heavy lifting** in structured domains (ArXiv, Science Papers) — adding contextualisation on top of a reranked pipeline adds near-zero improvement
- **The 67% headline** combines all improvements together and compares against vanilla embeddings with no BM25 and no reranking — the weakest possible baseline
- **The honest improvement** from contextualisation alone (holding reranking constant) is domain-dependent: large gains for narrative content, near-zero for structured scientific text

---

## Production Tips From the Blog

1. Use **prompt caching** when generating chunk contexts — the document only needs to be processed once, reducing cost to ~$1.02 per million document tokens
2. Use **Voyage or Gemini** embeddings — they outperformed others in the blog's experiments
3. Retrieve **top 150 candidates** before reranking, then keep top 20 for the LLM
4. If your knowledge base is **under 200,000 tokens**, just put the whole thing in the prompt — no RAG needed